# First Day of the Week Around the World

A pandas analysis of which weekday calendars treat as the start of the week, and how that pattern looks by **territory count**, **people**, and **region**.

Calendars do not agree on whether the week begins on Sunday or Monday. A smaller set of countries start on Saturday, and one starts on Friday. This notebook uses public locale and population data to answer a set of questions about those conventions.

**Datasets**
- `first-day-of-week.csv` — territory, ISO alpha-3 code, first weekday, measurement units, paper size (from [CLDR](https://unicode.org/cldr/) territory information)
- `population.csv` — 2020 population in millions (UN via Gapminder)
- `four-regions.csv` — Gapminder four-region grouping: africa, americas, asia, europe

## Setup

In [ ]:
# Google Colab only: uncomment to upload the three CSV files.
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

weekdays = pd.read_csv("first-day-of-week.csv")
population = pd.read_csv("population.csv")
regions = pd.read_csv("four-regions.csv")

day_order = ["fri", "sat", "sun", "mon"]
day_labels = {"fri": "Friday", "sat": "Saturday", "sun": "Sunday", "mon": "Monday"}

print("Weekday file:", weekdays.shape)
print("Population file:", population.shape)
print("Regions file:", regions.shape)
display(weekdays.head())
display(population.head())
display(regions.head())

Join population and region onto each territory. Left joins keep every territory from the weekday file so missing coverage is visible later.

In [ ]:
world = (
    weekdays.merge(population, on="alpha3", how="left")
    .merge(regions, on="alpha3", how="left")
)
world["first_day_label"] = world["first_day"].map(day_labels)
world.head()

---
## Question 1
How many territories start the week on Friday, Saturday, Sunday, and Monday?

In [ ]:
territory_counts = (
    world["first_day"]
    .value_counts()
    .reindex(day_order)
    .rename(index=day_labels)
    .to_frame("territories")
)
territory_counts["percent"] = 100 * territory_counts["territories"] / territory_counts["territories"].sum()
territory_counts

**Answer:** Most territories use Monday (186 of 257, about 72%). Sunday is next (55). Saturday is used in 15 territories, and Friday in 1 (the Maldives).

---
## Question 2
How many people start the week on each of those days?

Population is in millions (2020). Territories without a population row are excluded from the sums.

In [ ]:
people_by_day = (
    world.dropna(subset=["population"])
    .groupby("first_day", as_index=False)["population"]
    .sum()
)
people_by_day["first_day"] = pd.Categorical(people_by_day["first_day"], categories=day_order, ordered=True)
people_by_day = people_by_day.sort_values("first_day")
people_by_day["day"] = people_by_day["first_day"].map(day_labels)
people_by_day["percent_of_covered_population"] = (
    100 * people_by_day["population"] / people_by_day["population"].sum()
)
people_by_day[["day", "population", "percent_of_covered_population"]]

**Answer:** By people, Sunday and Monday are close. About 3.84 billion start on Sunday and 3.60 billion on Monday. Saturday covers about 432 million people; Friday covers 0.5 million (Maldives). Territory count favors Monday; population is split more evenly because India, the United States, Indonesia, Pakistan, Brazil, and Mexico use Sunday, while China (the largest country here) uses Monday.

---
## Question 3
Which of the four regions predominantly start the week on Sunday? On Monday? Are any regions split?

In [ ]:
region_days = (
    world.dropna(subset=["four_regions"])
    .pivot_table(
        index="four_regions",
        columns="first_day",
        values="territory",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=day_order)
    .rename(columns=day_labels)
)
region_days["total"] = region_days.sum(axis=1)
region_share = region_days.div(region_days["total"], axis=0).drop(columns="total") * 100

print("Territory counts")
display(region_days)
print("Share of territories in each region (%)")
display(region_share.round(1))

**Answer:**
- **Europe** is almost entirely Monday (46 of 48).
- **Africa** is mostly Monday (43 of 54), with some Saturday and Sunday.
- **The Americas** lean Sunday (20 vs 15 Monday).
- **Asia** is the most mixed: Monday 27, Sunday 21, Saturday 10, Friday 1.

---
## Question 4
If we weight by population instead of counting territories, which weekday wins in each region?

In [ ]:
region_people = (
    world.dropna(subset=["four_regions", "population"])
    .pivot_table(
        index="four_regions",
        columns="first_day",
        values="population",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=day_order)
    .rename(columns=day_labels)
)
region_people["total"] = region_people.sum(axis=1)
region_people_share = region_people.div(region_people["total"], axis=0).drop(columns="total") * 100

print("Population (millions)")
display(region_people.round(2))
print("Share of people in each region (%)")
display(region_people_share.round(1))
print("Winning weekday by people:")
print(region_people.drop(columns="total").idxmax(axis=1))

**Answer:** Weighting by people pulls **Asia toward Sunday** (India, Indonesia, Pakistan, Bangladesh) even though China is Monday-first. **The Americas** become strongly Sunday because of the United States, Brazil, and Mexico. **Europe** and **Africa** stay Monday-first; Africa still has a sizable Saturday-first population (Egypt and neighbors).

---
## Question 5
Which territories start the week on Friday or Saturday?

In [ ]:
weekend_start = (
    world.loc[world["first_day"].isin(["fri", "sat"]), ["territory", "alpha3", "first_day", "four_regions", "population"]]
    .assign(first_day=lambda d: d["first_day"].map(day_labels))
    .sort_values(["first_day", "territory"])
)
weekend_start

**Answer:** Only the Maldives starts on Friday. Saturday-first territories are mostly in North Africa and the Middle East (Afghanistan, Algeria, Bahrain, Djibouti, Egypt, Iran, Iraq, Jordan, Kuwait, Libya, Oman, Qatar, Sudan, Syria, UAE).

---
## Question 6
Do measurement units (metric vs US/UK) line up with first weekday?

In [ ]:
units_vs_day = pd.crosstab(world["units"], world["first_day"]).reindex(columns=day_order).rename(columns=day_labels)
display(units_vs_day)

print("Territories that are not purely metric:")
world.loc[world["units"] != "metric", ["territory", "units", "first_day", "paper"]]

**Answer:** Almost every territory is metric. The few US-customary territories (Bahamas, Belize, United States, and a handful of others) tend to use Sunday and US-Letter paper. Units are a weak signal compared with region.

---
## Question 7
How does paper size (A4 vs US-Letter) relate to first weekday and region?

In [ ]:
paper_vs_day = pd.crosstab(world["paper"], world["first_day"]).reindex(columns=day_order).rename(columns=day_labels)
paper_vs_region = pd.crosstab(world["paper"], world["four_regions"].fillna("unmapped"))

display(paper_vs_day)
display(paper_vs_region)

world.loc[world["paper"] == "US-Letter", ["territory", "four_regions", "first_day", "units"]].sort_values("territory")

**Answer:** A4 is the global default (243 territories). US-Letter is concentrated in the Americas and a few Pacific/Caribbean territories, and those places usually start the week on Sunday.

---
## Question 8
How complete are the joins? How many territories have no population or no region?

In [ ]:
coverage = pd.DataFrame(
    {
        "territories": [len(world)],
        "with_population": [world["population"].notna().sum()],
        "missing_population": [world["population"].isna().sum()],
        "with_region": [world["four_regions"].notna().sum()],
        "missing_region": [world["four_regions"].isna().sum()],
    }
)
display(coverage)

unmatched = world.loc[
    world["population"].isna() | world["four_regions"].isna(),
    ["territory", "alpha3", "first_day", "population", "four_regions"],
].sort_values("territory")
print(f"Unmatched or incomplete rows: {len(unmatched)}")
unmatched.head(20)

**Answer:** 61 territories have no population and/or no Gapminder region. They are mostly small islands, overseas territories, and special codes (Antarctica, Åland, American Samoa, and similar). Question 1 uses all 257 locales. Questions about people only use the 196 rows with population.

---
## Question 9
What is the first weekday for the 15 most populous territories?

In [ ]:
top15 = (
    world.dropna(subset=["population"])
    .nlargest(15, "population")
    [["territory", "alpha3", "four_regions", "first_day", "population"]]
    .assign(first_day=lambda d: d["first_day"].map(day_labels))
)
display(top15)
print("First-day counts among the top 15:")
print(top15["first_day"].value_counts())

**Answer:** Among the 15 largest territories in this file: China, Nigeria, Russia, and Vietnam start on Monday; Egypt starts on Saturday; the other 10 (India, United States, Indonesia, Pakistan, Brazil, Bangladesh, Mexico, Japan, Ethiopia, Philippines) start on Sunday. A few very large Sunday-first countries are enough to offset China on the global population totals.

---
## Question 10
For each region, what share of people start the week on Sunday versus Monday? Which region is the most divided?

In [ ]:
sun_mon = world.dropna(subset=["four_regions", "population"]).copy()
sun_mon = sun_mon[sun_mon["first_day"].isin(["sun", "mon"])]

split = (
    sun_mon.pivot_table(
        index="four_regions",
        columns="first_day",
        values="population",
        aggfunc="sum",
        fill_value=0,
    )
    .rename(columns=day_labels)
)
split["sunday_share_pct"] = 100 * split["Sunday"] / (split["Sunday"] + split["Monday"])
split["closeness_to_50"] = (split["sunday_share_pct"] - 50).abs()
split.sort_values("closeness_to_50")

**Answer:** **Asia** is the most divided (about 60% Sunday / 40% Monday by people). The Americas are about 88% Sunday. Africa is about 24% Sunday (Monday-leaning). Europe is about 1% Sunday — the least divided.

---
## Question 11
Build a compact summary table: territories, people, and people per territory for each first weekday.

In [ ]:
summary = (
    world.groupby("first_day")
    .agg(
        territories=("territory", "count"),
        territories_with_population=("population", "count"),
        people_millions=("population", "sum"),
    )
    .reindex(day_order)
    .rename(index=day_labels)
)
summary["avg_population_millions"] = summary["people_millions"] / summary["territories_with_population"]
summary

**Answer:** Sunday-first territories are fewer than Monday-first ones, but they have a higher average population. That is why a Monday-majority map can still be a Sunday-plurality (or near-tie) world by people.

---
## Question 12
Visualize territory counts and population by first weekday, and the regional mix.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

territory_counts["territories"].plot(kind="bar", ax=axes[0], color="#3b6d9a", rot=0)
axes[0].set_title("Territories by first weekday")
axes[0].set_ylabel("Number of territories")
axes[0].set_xlabel("")

people_plot = people_by_day.set_index("day")["population"]
people_plot.plot(kind="bar", ax=axes[1], color="#c46b3a", rot=0)
axes[1].set_title("People by first weekday (millions)")
axes[1].set_ylabel("Population (millions)")
axes[1].set_xlabel("")

region_days.drop(columns="total").plot(kind="bar", stacked=True, ax=axes[2], rot=0, colormap="tab10")
axes[2].set_title("Territories by region and weekday")
axes[2].set_xlabel("")
axes[2].set_ylabel("Territories")
axes[2].legend(title="", fontsize=8)

plt.tight_layout()
plt.show()

---
## Takeaways

1. **Most places use Monday**; most of the mapped population is split between **Sunday and Monday**.
2. **Europe** is Monday-first. **The Americas** are Sunday-first, especially by population. **Asia** is mixed. **Africa** is Monday-first with a Saturday belt in the north.
3. Friday is rare (Maldives). Saturday clusters in North Africa and the Middle East.
4. Population and region files do not cover every CLDR territory, so people-based answers use a smaller country set than the locale list.
5. Measurement units and paper size mostly follow the same US vs rest-of-world split as Sunday vs Monday, but region explains the weekday pattern better.